# Qwen3.5-0.8B Continued Pre-Training (CPT) Pipeline
### Google Colab Training Notebook (Phase 1)

This notebook runs full-parameter Continued Pre-Training on **Qwen3.5-0.8B-Base** (~1B target tokens) using a curated mixture of:
- 35% Stack v3 Code (`HuggingFaceCode/stack-v3-train`)
- 20% Stack v3 Documentation (`.md`, `.rst`, `README`)
- 20% The Vault (`Fsoft-AIC/the-vault-function`)
- 15% FineWeb-HQ (`epfml/FineWeb-HQ`)
- 10% OpenWebMath (`open-web-math/open-web-math`)

## 1. Hardware & Environment Check

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")

## 2. Mount Google Drive & Set Up Project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create project directory on Google Drive for persistence
!mkdir -p /content/drive/MyDrive/qwen-coder/checkpoints
!mkdir -p /content/drive/MyDrive/qwen-coder/data
!mkdir -p /content/drive/MyDrive/qwen-coder/logs

## 3. Install Dependencies

In [ ]:
!pip install -q --upgrade transformers datasets accelerate peft datasketch xxhash pyyaml rich huggingface_hub

## 4. Hugging Face Authentication (Optional for Hub Uploads)

In [ ]:
from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")

## 5. Verify Base Model Architecture (Text-Only / Strip Vision)

In [ ]:
from transformers import AutoTokenizer, Qwen3_5ForCausalLM

model_id = "Qwen/Qwen3.5-0.8B-Base"
print(f"Loading {model_id} as text-only Qwen3_5ForCausalLM...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = Qwen3_5ForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto"
)

total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Loaded text model with {total_params:,} parameters ({total_params/1e9:.2f}B)")

# Test quick generation
inputs = tokenizer("def quicksort(arr):\n", return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(out[0], skip_special_tokens=True))

## 6. Process Data / Prepare Training Shards
Run data processing pipeline directly or stream from preprocessed dataset shards.

In [ ]:
# Run data pipeline to generate packed shards
!python scripts/03_process_data.py --output-dir /content/drive/MyDrive/qwen-coder/data

## 7. Run Full CPT Training
Launches training with gradient checkpointing, cosine LR schedule, and automatic Drive checkpointing.

In [ ]:
!python scripts/05_train_cpt.py --data-dir /content/drive/MyDrive/qwen-coder/data

## 8. Benchmark Evaluation & Base vs CPT Comparison

In [ ]:
!python scripts/06_evaluate.py --compare --base Qwen/Qwen3.5-0.8B-Base --cpt /content/drive/MyDrive/qwen-coder/checkpoints/final --output /content/drive/MyDrive/qwen-coder/logs/comparison.json